In [ ]:
class Node:
    def __init__(self, state, parent=None, cost=0, depth=0):
        self.state = state
        self.parent = parent
        self.cost = cost
        self.depth = depth
    
def get_path(node):
    path = []
    while node:
        path.append(node.state)
        node = node.parent
    return path[::-1]

In [ ]:
import heapq

def greedy_bfs(graph, heuristic, start, goal):
    pq = []
    heapq.heappush(pq, (heuristic[start], Node(start)))
    visited = set()

    while pq:
        _, current_node = heapq.heappop(pq)
        current_state = current_node.state

        if current_state == goal:
            return get_path(current_node)
        
        visited.add(current_state)

        for neighbor_state, _ in graph[current_state]:
            if neighbor_state not in visited: # Pre-evaluation (Checking before pushing)
                neighbor_node = Node(neighbor_state, current_node)
                heapq.heappush(pq, (heuristic[neighbor_state], neighbor_node))

    return None
        

In [ ]:
import heapq

def astar(graph, heuristic, start, goal):
    pq = []
    heapq.heappush(pq, (heuristic[start], Node(start, cost=0))) # This was mistake identified my me 😀
    visited = {}

    while pq:
        f_cost, current_node = heapq.heappop(pq)
        current_state = current_node.state

        if current_state == goal:
            return get_path(current_node)
        
        if current_state not in visited or f_cost < visited[current_state]: # Post-evaluation (Checking after popping) MUST IN UCS AND A*
            visited[current_state] = f_cost

            for neighbor_state, edge_cost in graph[current_state]:
                g_n = current_node.cost + edge_cost
                h_n = heuristic[neighbor_state]
                f_n = g_n + h_n

                neighbor_node = Node(neighbor_state, current_node, g_n)
                heapq.heappush(pq, (f_n, neighbor_node))
    
    return None

In [10]:
class GameNode:
    def __init__(self, value=None, children=None):
        self.value = value
        self.children = children

In [11]:
def minimax(node, ismax):
    # Terminal Node
    if not node.children:
        return node.value
    
    if ismax:
        best = float('-inf')
        for child in node.children:
            value = minimax(child, False)
            best = max(best, value)
        return best
    else:
        best = float('inf')
        for child in node.children:
            value = minimax(child, True)
            best = min(best, value)
        return best
    
leaf1 = GameNode(3)
leaf2 = GameNode(5)
leaf3 = GameNode(2)
leaf4 = GameNode(9)

min1 = GameNode(children=[leaf1, leaf2])
min2 = GameNode(children=[leaf3, leaf4])

root = GameNode(children=[min1, min2])

print("Optimal Value: ", minimax(root, True))
 

Optimal Value:  3


# CSV Student Performance Analyzer

In [33]:
import csv 

# Custom Error class
class NegativeMarkError(Exception):
    pass

student_records = {}

try :
    with open("students.csv", "r") as file:
        reader = csv.reader(file)
        students = list(reader)
        del students[0]

        for student in students:
            name = student[0]
            total_marks, sub_count = 0, 0
            # print(student)

            for mark in student[1:]:
                try:
                    mark = float(mark)
                    if mark < 0:
                        raise NegativeMarkError(f"Found negative marks for {name}")
                    total_marks += mark
                    sub_count += 1
                except NegativeMarkError as e:
                    print(f"Skipping student's record: {e}")
                except ValueError:
                    print(f"Skipping student's record: Found invalid number format for {name}")

            avg_marks = total_marks / sub_count
            student_records[name] = [total_marks, avg_marks]
except FileNotFoundError:
    print("Error: The file you're trying to access does not exist.")

print(student_records)

Skipping student's record: Found negative marks for Hassan
Skipping student's record: Found negative marks for Usman
Skipping student's record: Found invalid number format for Bilal
{'Ali': [263.0, 87.66666666666667], 'Ahmad': [207.0, 69.0], 'Sara': [281.0, 93.66666666666667], 'Zain': [183.0, 61.0], 'Fatima': [269.0, 89.66666666666667], 'Hassan': [153.0, 76.5], 'Ayesha': [285.0, 95.0], 'Usman': [142.0, 71.0], 'Maryam': [248.0, 82.66666666666667], 'Bilal': [147.0, 73.5], 'Khadija': [268.0, 89.33333333333333], 'Hamza': [199.0, 66.33333333333333], 'Noor': [269.0, 89.66666666666667], 'Ibrahim': [240.0, 80.0], 'Hina': [252.0, 84.0]}


# Inventory Management System

In [ ]:
import json
import os

class Product:
    def __init__(self, name, price, quantity):
        self.name = name

        if price <= 0:
            raise ValueError("Price must be positive.")
        self.price = price

        if quantity < 0:
            raise ValueError("Quantity cannot be negative.")
        self.__quantity = quantity

    def add_stock(self, amount):
        if amount <= 0:
            raise ValueError("Restock amount must be positive.")
        self.__quantity += amount

    def sell_stock(self, amount):
        if amount <= 0:
            raise ValueError("Sell amount must be positive.")
        if amount > self.__quantity:
            raise ValueError("Insufficient stock.")
        self.__quantity -= amount

    def get_quantity(self):
        return self.__quantity


class Inventory:
    def __init__(self):
        self.products = {}

    def add_product(self, name, price, quantity):
        if name in self.products:
            raise KeyError("Product already exists.")
        self.products[name] = Product(name, price, quantity)

    def sell_product(self, name, quantity):
        if name not in self.products:
            raise KeyError("Product not found.")
        self.products[name].sell_stock(quantity)

    def restock_product(self, name, quantity):
        if name not in self.products:
            raise KeyError("Product not found.")
        self.products[name].add_stock(quantity)

    def display_inventory(self):
        if not self.products:
            print("Inventory empty.")
            return

        for p in self.products.values():
            print(p.name, p.price, p.get_quantity())

    def save_to_file(self, filename):
        data = []
        for p in self.products.values():
            data.append({
                "name": p.name,
                "price": p.price,
                "quantity": p.get_quantity()
            })

        with open(filename, "w") as f:
            json.dump(data, f, indent=4)

    def load_from_file(self, filename):
        if not os.path.exists(filename):
            return

        with open(filename, "r") as f:
            data = json.load(f)

        for item in data:
            p = Product(item["name"], item["price"], item["quantity"])
            self.products[item["name"]] = p


# ---------------- USER INTERFACE ----------------

inventory = Inventory()
db_file = "inventory_data.json"

inventory.load_from_file(db_file)

while True:
    print("\n[1] Add [2] Sell [3] Restock [4] View [5] Exit")
    cmd = input("Select an option: ")

    try:
        if cmd == '1':
            name = input("Product Name: ")
            price = float(input("Price: "))
            qty = int(input("Quantity: "))
            inventory.add_product(name, price, qty)

        elif cmd == '2':
            name = input("Product Name: ")
            qty = int(input("Quantity to sell: "))
            inventory.sell_product(name, qty)

        elif cmd == '3':
            name = input("Product Name: ")
            qty = int(input("Quantity to add: "))
            inventory.restock_product(name, qty)

        elif cmd == '4':
            inventory.display_inventory()

        elif cmd == '5':
            inventory.save_to_file(db_file)
            print("Inventory saved. Exiting.")
            break

        else:
            print("Invalid option.")

        inventory.save_to_file(db_file)

    except ValueError as e:
        print("Input Error:", e)

    except KeyError as e:
        print("Record Error:", e)

    except Exception as e:
        print("System Error:", e)


[1] Add [2] Sell [3] Restock [4] View [5] Exit
Invalid option.

[1] Add [2] Sell [3] Restock [4] View [5] Exit

[1] Add [2] Sell [3] Restock [4] View [5] Exit
Record Error: 'Product not found.'

[1] Add [2] Sell [3] Restock [4] View [5] Exit
Invalid option.

[1] Add [2] Sell [3] Restock [4] View [5] Exit


# Implement Greedy and A* Search

In [44]:
graph = {
    'A': [('B', 1), ('C', 4)],
    'B': [('D', 2), ('E', 5)],
    'C': [('F', 1)],
    'D': [],
    'E': [],
    'F': []
}
heuristic = {
    'A': 7,
    'B': 6,
    'C': 2,
    'D': 1,
    'E': 3,
    'F': 0
}
print("Greedy: ", greedy_bfs(graph, heuristic, 'A', 'F'))
print("A*: ", astar(graph, heuristic, 'A', 'F'))

Greedy:  ['A', 'C', 'F']
A*:  ['A', 'C', 'F']


Which algorithm found the better path?

In many complex graphs, A* finds the better (shorter) path. In the simple example above, they might find the same path, but Greedy is "short-sighted."
    Greedy is "greedy": It makes the best choice at the immediate moment. If a path looks short but actually leads to a massive mountain range (high edge weight) later on, Greedy will still take it.
    A balances both:* It looks at how much work it has already done (g) PLUS how much is left (h). This prevents it from taking a path that "looks" short but is actually very long in reality.

# Minimax Tree Evaluation

In [45]:
leaf1 = GameNode(3)
leaf2 = GameNode(12)
leaf3 = GameNode(8)
leaf4 = GameNode(2)
leaf5 = GameNode(4)
leaf6 = GameNode(6)

min1 = GameNode(children=[leaf1, leaf2])
min2 = GameNode(children=[leaf3, leaf4])
min3 = GameNode(children=[leaf5, leaf6])

root = GameNode(children=[min1, min2, min3])

print("Optimal Value: ", minimax(root, True))

Optimal Value:  4


Min will choose the value of 3, 2, 4 from pairs (3,12), (8,2), (4,6) respectively.
Also Max will choose the maximum value it will get from its child nodes that ran Min which are max(3,2,4) = 4 as ans.